# Week 6 — Theory
## Computer vision explainability and layer-wise visualization

Last week we attributed predictions to **input features**. This week we attribute them
to **internal features** — feature maps in a CNN, attention patterns in a Vision
Transformer.

Why is this its own week?

- For deep vision models, input-pixel attribution is *very* noisy. Smoothing it at the
  level of feature maps produces a more useful chart.
- For ViTs, the model itself exposes an interpretable internal structure (attention),
  which gives us a different family of methods.
- The whole subfield has a recurring pitfall: visualizations that **look plausible**
  but do not actually **reflect the decision**. This week's faithfulness metrics are
  how we distinguish the two.

Coverage:

1. The **Grad-CAM family** — Grad-CAM, Grad-CAM++, Score-CAM, EigenCAM. How they
   compute a class-discriminative spatial map.
2. **Attention rollout** for Vision Transformers.
3. **Faithfulness metrics** — deletion / insertion AUC, average drop, sufficient /
   necessary masks.
4. **Plausibility vs. faithfulness** — and the experiment design that separates them.


## 1. The Grad-CAM family

### Grad-CAM — derivation

Let $A^k \in \mathbb{R}^{H \times W}$ be the $k$-th feature map at a chosen
convolutional layer, and let $y^c$ be the pre-softmax score for class $c$. Define the
neuron-importance weight:

$$
\alpha_k^c = \frac{1}{H \cdot W} \sum_{i, j} \frac{\partial y^c}{\partial A^k_{ij}}
$$

— that is, the global-average-pooled gradient of the class score with respect to each
position in the feature map. The Grad-CAM map is:

$$
L^c_{\text{Grad-CAM}} = \mathrm{ReLU}\!\left(\sum_k \alpha_k^c A^k\right)
$$

In English: weight each feature map by how much it influences the class score on
average, sum them up, throw away negatives, upsample to image size.

**Why the ReLU?** Grad-CAM is defined to be **class-discriminative**: we only want
pixels that argue *for* the class, not against it. Negative contributions argue against
and are zeroed.

**Where do you apply it?** At the **last convolutional layer**, just before global
average pooling. That layer has the highest semantic content while still preserving
spatial structure.

### Grad-CAM++ — improving over-localization

Grad-CAM can fail when the class object appears in multiple, well-separated regions of
the image (think: two dogs). The original definition tends to highlight only one. Grad-CAM++
(Chattopadhay et al., 2018) modifies the weight calculation to be a weighted average of
positive-gradient contributions:

$$
\alpha_k^c = \sum_{ij}
\frac{\partial^2 y^c / \partial (A^k_{ij})^2}{2 \cdot \partial^2 y^c / \partial (A^k_{ij})^2 + \sum_{ab} A^k_{ab} \cdot \partial^3 y^c / \partial (A^k_{ij})^3}
\cdot \mathrm{ReLU}\!\left(\frac{\partial y^c}{\partial A^k_{ij}}\right)
$$

The expression is uglier; the effect is to localize multiple object instances of the
same class.

### Score-CAM — gradient-free

Score-CAM (Wang et al., 2020) avoids gradients entirely. For each feature map $A^k$:

1. Upsample $A^k$ to image size, normalize to $[0, 1]$, multiply elementwise with the
   input.
2. Feed the masked image into the model, record the class score.

The Score-CAM map is the sum of feature maps weighted by these scores. **No gradients
needed** — useful when you have access to the model's output but not its internals
(e.g. an API). Slow: one forward pass per feature map.

### EigenCAM — the principal component

EigenCAM (Muhammad & Yeasin, 2020) computes the leading singular vector of the feature
map tensor at the chosen layer. It is **not class-discriminative** — it shows the
dominant spatial pattern at that layer regardless of the predicted class — which makes
it a useful sanity baseline. If your Grad-CAM looks identical to your EigenCAM, the
"explanation" is not actually class-specific.

### Practical summary

| Method      | Class-discriminative? | Gradient-based? | Multi-instance? | Cost |
|-------------|:--------------------:|:---------------:|:---------------:|:----:|
| Grad-CAM    | ✓                    | ✓               | ✗ (often)       | cheap |
| Grad-CAM++  | ✓                    | ✓               | ✓               | cheap |
| Score-CAM   | ✓                    | ✗               | ✓               | slow |
| EigenCAM    | ✗ (baseline)         | ✗               | -               | cheap |


## 2. Attention rollout for Vision Transformers

ViTs split the input into patches, project each to a token, and run self-attention. The
attention weights $A^{(\ell)} \in \mathbb{R}^{T \times T}$ at layer $\ell$ tell us how
strongly each token attends to each other token.

**Naive idea.** Look at the attention from the `[CLS]` token at the last layer. This
sometimes works, sometimes does not — late-layer attention is mixed across many earlier
layers' contributions.

**Attention rollout** (Abnar & Zuidema, 2020) addresses this by tracing the attention
*through* the layers. For each layer, average the attention across heads and add the
residual connection's identity matrix:

$$
\bar A^{(\ell)} = \frac{1}{2}\!\left(\mathrm{mean}_{h}\,A^{(\ell, h)} + I\right)
$$

Row-normalise, then multiply layer-wise:

$$
R = \bar A^{(L)} \cdot \bar A^{(L-1)} \cdots \bar A^{(1)}
$$

The row of $R$ corresponding to the `[CLS]` token, reshaped into a spatial grid, is the
**rollout attribution**. It tells you which input patches the classification token
ultimately attended to, accounting for the full attention path.

### Caveats specific to attention

Attention is **not** automatically an explanation. Two known issues:

- **Attention is not unique.** Many attention patterns produce the same model output.
- **Attention can be high on irrelevant tokens** if the residual stream and MLP do the
  real work. A `[CLS]` token attending to a feature does not mean the feature decided
  the prediction — only that the model looked.

These are reasons to **always pair attention rollout with a faithfulness check**, the
same as for any other attribution method.


## 3. Faithfulness metrics

We can score an attribution method on a held-out set with two cheap, model-agnostic
metrics.

### Deletion AUC

Start with the original image. Iteratively **remove** (e.g. replace with the mean
pixel or a blur) the top-scoring K % of pixels by the attribution map, recompute the
class probability, and plot the probability vs. fraction removed.

A faithful attribution gives a **steep drop** — removing the highlighted pixels really
does destroy the prediction. AUC under this curve is the **deletion score**; lower is
better.

### Insertion AUC

The mirror image. Start with a blurred / empty image. Iteratively **insert** the
top-scoring K % of pixels back, recompute the probability. Faithful attributions give a
**steep climb**; AUC is the **insertion score**; higher is better.

### Average drop and sufficient mask

Average drop: $\max(0, f(x) - f(x \odot M))$ averaged over the test set, where $M$ is
a binary mask kept around the top-K attribution pixels. Lower = more faithful.

Sufficient mask: smallest mask such that $f(x \odot M) \ge \tau \cdot f(x)$. Useful
for paper figures because it gives an interpretable "the model needed *this many*
pixels".


## 4. Plausibility vs. faithfulness

A common failure mode: an attribution method produces maps that look like *where you
would expect a human to look* — face, eyes, salient object — and is therefore reported
as "validating that the model attends to the right region".

But the model is not a human. It might be using texture cues, background context, or
other shortcuts. An attribution method that smooths whatever it computes toward
human-salient regions will produce **plausible** maps regardless of what the model
actually does. These are the methods that fail Adebayo et al.'s sanity check from week 5.

**The discipline of faithfulness:**

1. Always pair the qualitative map with a quantitative faithfulness score on a held-out
   set.
2. Run the **parameter-randomization sanity check** on your method.
3. When the deletion/insertion curve is shallow, do not report the map as an
   explanation — report it as "the model attended *here* but our attribution is not
   discriminating well", which is honest and informative.


## 5. When to use what

| Question                                                                    | Reach for                          |
|-----------------------------------------------------------------------------|------------------------------------|
| "Which region of the image drove this CNN's prediction?"                    | Grad-CAM++ on the last conv layer  |
| "Same, but the model is a black-box API"                                    | Score-CAM, RISE                    |
| "Which patches did this ViT use?"                                           | Attention rollout                  |
| "Is the attribution method actually faithful, or just plausible?"           | Deletion / insertion AUC           |
| "Does the network represent a particular concept somewhere internally?"     | Concept activation vectors (TCAV)  |
| "Per-pixel attribution on a small differentiable model"                     | Integrated Gradients (week 5)      |

For most papers a defensible explanation section combines: **(a)** a qualitative
visualisation (Grad-CAM or rollout); **(b)** a quantitative faithfulness score;
**(c)** a sanity-check experiment on a randomly-initialized model. Without all three,
the chart is a hypothesis, not evidence.


## Summary

- **Grad-CAM** is the workhorse for CNNs: cheap, axiomatic enough to be defensible,
  and class-discriminative. Use **Grad-CAM++** when multiple instances appear in the
  same image.
- **Attention rollout** is the equivalent default for ViTs, with the caveat that
  attention is not automatically an explanation.
- **Faithfulness metrics** (deletion / insertion AUC) are how you separate explanations
  that work from explanations that look nice.
- The most common mistake in this corner of the field is **conflating plausibility
  with faithfulness**. Always check the latter.

In the lab we apply Grad-CAM and attention rollout to a small set of images, then run
a deletion-AUC evaluation to see whether the visualisations are doing what they claim.
